# 🧹 Data Cleaning Pipeline

This notebook walks through a step-by-step data cleaning process for a product dataset.
It covers:
- Loading the raw dataset
- Stripping HTML tags from the `body_html` column and renaming it to `description`
- Filtering the `tags` column to retain only relevant tag prefixes
- Renaming the `product_type` column to `author`
- Converting the `id` column data type from numeric to text (string)
- Saving the cleaned dataset to a new CSV file

## Step 0 — Import Libraries

We import `pandas` for data manipulation and `re` (regular expressions) for HTML tag removal.

In [1]:
import pandas as pd
import re

print("Libraries imported successfully.")

Libraries imported successfully.


## Step 1 — Load the Dataset

Load the raw CSV file into a pandas DataFrame. Update the `FILE_PATH` variable to point to your actual file location.

We then preview the first few rows and check the relevant columns (`body_html`, `tags`, `product_type`) to confirm they loaded correctly.

In [2]:
# ── Configuration ────────────────────────────────────────────────────────────
FILE_PATH   = "raw_data.csv"   # ← update this path to your file
OUTPUT_PATH = "cleaned_dataset.csv"

# ── Load ─────────────────────────────────────────────────────────────────────
df = pd.read_csv(FILE_PATH)

print(f"Dataset loaded — {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\nColumns present: {df.columns.tolist()}")

# Preview the three columns we will work on
cols_of_interest = ["id", "body_html", "tags", "product_type"]
existing = [c for c in cols_of_interest if c in df.columns]
missing  = [c for c in cols_of_interest if c not in df.columns]

if missing:
    print(f"\n⚠️  Missing expected columns: {missing}")

df[existing].head(3)

Dataset loaded — 189,456 rows × 11 columns

Columns present: ['id', 'title', 'handle', 'vendor', 'product_type', 'created_at', 'updated_at', 'published_at', 'tags', 'body_html', 'source_url']


,id,body_html,tags,product_type
0,5.915860e+12,<p></p><div> Why have history's greatest min...,"BOOK, Class Code_104, Class_Business Finance &...","Holiday, Ryan"
1,5.915870e+12,<p></p><p> The year is 1984 and the city is ...,"Class Code_302, Class_ADULT FICTION, Departmen...",Haruki Murakami
2,5.915860e+12,<p><strong>The international bestselling YA th...,"BOOK, Class Code_232, Class_Young Adult, Depar...","Mcmanus, Karen"


## Step 2 — Clean the HTML Content and Rename to `description`

The `body_html` column contains raw HTML markup (e.g. `<p>`, `<div>`, `<strong>`, `<h5>`, `<br>` with inline styles and attributes). 

We use a regular expression to strip **all** HTML tags, then clean up any leftover whitespace so we are left with readable plain text.

Finally, we overwrite `body_html` with the cleaned text and rename the column to `description`.

In [3]:
def strip_html(raw: str) -> str:
    """Remove all HTML tags and normalise whitespace."""
    if not isinstance(raw, str):
        return ""                              # handle NaN / non-string values
    text = re.sub(r"<[^>]+>", " ", raw)       # replace every <tag …> with a space
    text = re.sub(r"\s+", " ", text).strip()  # collapse multiple spaces / newlines
    return text

# Apply the cleaner and overwrite the column in-place
df["body_html"] = df["body_html"].apply(strip_html)

# Rename body_html → description
df.rename(columns={"body_html": "description"}, inplace=True)

print("✅ HTML stripped and column renamed to 'description'.")
df["description"].head(3)

✅ HTML stripped and column renamed to 'description'.


0    Why have history's greatest minds—from George ...
1    The year is 1984 and the city is Tokyo. A youn...
2    The international bestselling YA thriller by a...
Name: description, dtype: object

## Step 3 — Filter the `tags` Column

Each row's `tags` field is a comma-separated string of values, many of which are irrelevant (e.g. brand tags, colour tags, etc.).

We keep **only** tags that start with one of three recognised prefixes:
- `Class Code_`
- `Class_`
- `Department_`

The filtered tags are joined back into a comma-separated string and written back to the same `tags` column, overwriting the original.

In [4]:
KEEP_PREFIXES = ("Class Code_", "Class_", "Department_")

def filter_tags(tag_string: str) -> str:
    """Keep only tags matching the allowed prefixes."""
    if not isinstance(tag_string, str) or tag_string.strip() == "":
        return ""                                   # handle NaN / empty cells
    tags = [t.strip() for t in tag_string.split(",")]
    kept = [t for t in tags if t.startswith(KEEP_PREFIXES)]
    return ", ".join(kept)

# Apply and overwrite the tags column
df["tags"] = df["tags"].apply(filter_tags)

print("✅ Tags filtered — only Class Code_, Class_, and Department_ tags retained.")
df["tags"].head(3)

✅ Tags filtered — only Class Code_, Class_, and Department_ tags retained.


0    Class Code_104, Class_Business Finance & Accou...
1    Class Code_302, Class_ADULT FICTION, Departmen...
2    Class Code_232, Class_Young Adult, Department_...
Name: tags, dtype: object

## Step 4 — Rename `product_type` to `author`

The `product_type` column is renamed to `author` to better reflect the data it holds in this context.
No transformations are made to the cell values themselves — only the column header changes.

In [5]:
df.rename(columns={"product_type": "author"}, inplace=True)

print("✅ Column 'product_type' renamed to 'author'.")
df["author"].head(3)

✅ Column 'product_type' renamed to 'author'.


0      Holiday, Ryan
1    Haruki Murakami
2     Mcmanus, Karen
Name: author, dtype: object

## Step 5 — Convert `id` Column to Text

By default, pandas reads numeric-looking columns like `id` as integers or floats.
Converting it to a string (`object`) data type ensures:
- Leading zeros are never silently dropped.
- The value is treated as an identifier, not a number to be summed or averaged.
- Downstream systems (e.g. databases, APIs) that expect a string ID receive the correct type.

We also verify the dtype before and after the conversion.

In [6]:
print(f"Before — 'id' dtype: {df['id'].dtype}")

# Convert id to string, filling any NaN with an empty string first
df["id"] = df["id"].fillna("").astype(str)

# If the column was numeric, astype(str) may produce '0' for NaN-filled rows;
# replace those placeholders back with an empty string.
df["id"] = df["id"].replace("nan", "")

print(f"After  — 'id' dtype: {df['id'].dtype}")
print("\n✅ 'id' column converted to text (string).")
df["id"].head(3)

Before — 'id' dtype: float64
After  — 'id' dtype: object

✅ 'id' column converted to text (string).


0    5915860000001.0
1    5915870000002.0
2    5915860000003.0
Name: id, dtype: object

## Step 6 — Remove Number/Date as `title`

What it does:

- `float(val)` catches whole numbers, decimals, and scientific notation like `7.7157E+12`
- `date_parse()` catches date formats like `1/7/1914 0:00` and `11-Sep-01`
- The extra `re.search(r"\d", val)` guard prevents plain words from accidentally matching as dates
- Filters in-place on df so it flows directly into the save step below it

In [7]:
from dateutil.parser import parse as date_parse

def is_number_or_date(value: str) -> bool:
    """Return True if the value is purely a number (int/float) or a date string."""
    val = str(value).strip()

    if val == "" or val.lower() == "nan":
        return False

    # Check if it's a pure number (int, float, scientific notation)
    try:
        float(val)
        return True
    except ValueError:
        pass

    # Check if it's a pure date string (e.g. "1/7/1914 0:00", "11-Sep-01")
    try:
        date_parse(val, fuzzy=False)
        # Extra guard: must contain a digit and NOT be a plain word
        if re.search(r"\d", val):
            return True
    except Exception:
        pass

    return False

# Identify the column to check — update if your column name differs
TARGET_COLUMN = "title"   # ← change to your actual column name if different

before = len(df)

df = df[~df[TARGET_COLUMN].apply(is_number_or_date)].reset_index(drop=True)

after = len(df)
print(f"✅ Removed {before - after:,} rows where '{TARGET_COLUMN}' was a number or date.")
print(f"   Rows remaining: {after:,}")

✅ Removed 58 rows where 'title' was a number or date.
   Rows remaining: 189,398


## Step 6 — Verify and Save the Cleaned Dataset

Before saving, we do a quick sanity check to confirm:
1. The target columns (`description`, `tags`, `author`, `id`) are present and have the correct types.
2. The old column names (`body_html`, `product_type`) are gone.

Then we export the cleaned DataFrame to `OUTPUT_PATH` without the default pandas index column.

In [8]:
# ── Sanity checks ─────────────────────────────────────────────────────────────
expected_new = ["description", "tags", "author", "id"]
expected_gone = ["body_html", "product_type"]

for col in expected_new:
    status = "✅" if col in df.columns else "❌ MISSING"
    dtype  = df[col].dtype if col in df.columns else "—"
    print(f"  {status}  '{col}' present  (dtype: {dtype})")

for col in expected_gone:
    status = "❌ STILL EXISTS" if col in df.columns else "✅"
    print(f"  {status}  '{col}' removed")

# ── Save ──────────────────────────────────────────────────────────────────────
df.to_csv(OUTPUT_PATH, index=False)
print(f"\n💾 Cleaned dataset saved to: {OUTPUT_PATH}")
print(f"   Final shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

  ✅  'description' present  (dtype: object)
  ✅  'tags' present  (dtype: object)
  ✅  'author' present  (dtype: object)
  ✅  'id' present  (dtype: object)
  ✅  'body_html' removed
  ✅  'product_type' removed

💾 Cleaned dataset saved to: cleaned_dataset.csv
   Final shape: 189,398 rows × 11 columns


## Final Preview

Display the first five rows of the four cleaned columns side-by-side as a final confirmation.

In [10]:
df.head()

,id,title,handle,vendor,author,created_at,updated_at,published_at,tags,description,source_url
0,5915860000001.0,"The Daily Stoic: 366 Meditations On Wisdom, Pe...",the-daily-stoic-366-meditations-on-wisdom-pers...,Portfolio US,"Holiday, Ryan",2020-12-15T19:14:53+08:00,2026-05-21T22:13:17+08:00,2020-12-29T05:40:55+08:00,"Class Code_104, Class_Business Finance & Accou...",Why have history's greatest minds—from George ...,https://mphonline.com/products/the-daily-stoic...
1,5915870000002.0,1Q84,1q84,Vintage,Haruki Murakami,2020-12-15T19:17:12+08:00,2026-05-21T22:13:17+08:00,2026-03-03T16:02:56+08:00,"Class Code_302, Class_ADULT FICTION, Departmen...",The year is 1984 and the city is Tokyo. A youn...,https://mphonline.com/products/1q84.json
2,5915860000003.0,One Of Us Is Lying by Karen Mcmanus,one-of-us-is-lying,Penguin UK,"Mcmanus, Karen",2020-12-15T19:14:49+08:00,2026-05-21T22:13:17+08:00,2020-12-16T13:12:59+08:00,"Class Code_232, Class_Young Adult, Department_...",The international bestselling YA thriller by a...,https://mphonline.com/products/one-of-us-is-ly...
3,5915870000004.0,Rich Dad's Retire Young Retire Rich: How to Ge...,rich-dads-retire-young-retire-rich-how-to-get-...,Perseus,"Kiyosaki, Robert T.",2020-12-15T19:16:53+08:00,2026-05-21T22:13:17+08:00,2020-12-29T05:40:52+08:00,"Class Code_105, Class_Business Finance & Accou...",Are your financial plans on the fast track or ...,https://mphonline.com/products/rich-dads-retir...
4,5915860000005.0,"Men Are from Mars, Women Are from Venus: The D...",men-are-from-mars-women-are-from-venus-the-def...,Harper,"Gray, John",2020-12-15T19:15:25+08:00,2026-05-21T22:13:18+08:00,2020-12-29T05:40:50+08:00,"Class Code_17, Class_Family & Relationship, De...",The legendary relationships guide that mothers...,https://mphonline.com/products/men-are-from-ma...
